# Model Evaluation: YOLOv8n + Scale-Adaptive CBAM
**Experiment ID:** `scale_adaptive_cbam`  
**Architecture:** Scale-Adaptive CBAM  
**Dataset:** Helmet Detection (2 classes: helmet, non-helmet)  
> Run this notebook to evaluate.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
EXPERIMENT_ID = "scale_adaptive_cbam"
WEIGHTS       = r"C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/models/best (scale_adaptive).pt"
DATA_YAML     = r"C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/data_big.yaml"
DATASET_ROOT  = r"C:\Users\soumy\OneDrive\Desktop\test(big)"
IMGSZ         = 640
BATCH         = 16
CONF_THRES    = 0.001
IOU_THRES     = 0.5
OUTPUT_DIR    = r"C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam"
CLASS_NAMES   = ["helmet", "non-helmet"]

In [2]:
!pip install ultralytics -q

In [3]:
import torch, torch.nn as nn, sys

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d(1)
        self.pool_m = nn.AdaptiveMaxPool2d(1)
        r = max(channels // reduction, 1)
        self.fc = nn.Sequential(nn.Conv2d(channels, r, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(r, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.fc(self.pool_h(x)) + self.fc(self.pool_m(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.conv(torch.cat([torch.mean(x,1,True), torch.max(x,1,True)[0]], 1)))

class ScaleAdaptiveCBAM(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.kernel_size = kernel_size
        self.ca = None; self.sa = None
    def _get_adaptive_reduction(self, channels):
        if channels <= 64: return 4
        elif channels <= 128: return 8
        elif channels <= 256: return 12
        else: return 16
    def forward(self, x):
        if self.ca is None:
            c = x.shape[1]
            reduction = self._get_adaptive_reduction(c)
            self.ca = ChannelAttention(c, reduction).to(x.device)
            self.sa = SpatialAttention(self.kernel_size).to(x.device)
        x = self.ca(x)
        x = self.sa(x)
        return x

import ultralytics.nn.tasks as tasks, ultralytics.nn.modules as modules
for cls in [ScaleAdaptiveCBAM, ChannelAttention, SpatialAttention]:
    setattr(tasks, cls.__name__, cls)
    setattr(modules, cls.__name__, cls)
    for s in ("block","conv","head"):
        if hasattr(modules,s): setattr(getattr(modules,s), cls.__name__, cls)
    sys.modules['__main__'].__dict__[cls.__name__] = cls
torch.serialization.add_safe_globals([ScaleAdaptiveCBAM, ChannelAttention, SpatialAttention])
print("[OK] ScaleAdaptiveCBAM registered")

[OK] ScaleAdaptiveCBAM registered


In [4]:
import os, json, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from ultralytics import YOLO

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[OK] Output directory: {OUTPUT_DIR}")
print(f"[OK] Loading weights : {WEIGHTS}")
model = YOLO(WEIGHTS)


[OK] Output directory: C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam
[OK] Loading weights : C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/models/best (scale_adaptive).pt


## 1 - Validation on Test Split

In [5]:
test_results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    conf=CONF_THRES,
    iou=IOU_THRES,
    save_json=True,
    plots=True,
    project=OUTPUT_DIR,
    name="test",
    exist_ok=True,
    verbose=True,
)

print("\n[Test] Raw metrics object:", test_results)

Ultralytics 8.4.6  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)


scale_adaptive_cbam summary (fused): 97 layers, 3,031,932 parameters, 0 gradients, 8.1 GFLOPs


val: Fast image access  (ping: 0.40.1 ms, read: 34.114.8 MB/s, size: 34.6 KB)


val: Scanning C:\Users\soumy\OneDrive\Desktop\test(big)\labels.cache... 1766 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1766/1766 211.6Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 1% ──────────── 1/111 3.0s/it 0.9s<5:27

...

            non-helmet        339       1803      0.944       0.94      0.972      0.669


Speed: 1.4ms preprocess, 5.2ms inference, 0.0ms loss, 1.5ms postprocess per image


Saving C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam\test\predictions.json...


Results saved to C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam\test



[Test] Raw metrics object: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002827D446120>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046

## 2 - Extract and Display Metrics

In [6]:
def extract_metrics(results, split_name):
    box = results.box
    metrics = {
        "experiment": EXPERIMENT_ID, "split": split_name,
        "mAP50": round(float(box.map50), 4), "mAP50_95": round(float(box.map), 4),
        "precision": round(float(box.mp), 4), "recall": round(float(box.mr), 4),
        "f1": round(float(np.mean(box.f1)), 4),
    }
    for i, cls_name in enumerate(CLASS_NAMES):
        try:
            metrics[f"AP50_{cls_name}"] = round(float(box.ap50[i]), 4)
            metrics[f"AP50_95_{cls_name}"] = round(float(box.ap[i]), 4)
            metrics[f"P_{cls_name}"] = round(float(box.p[i]), 4)
            metrics[f"R_{cls_name}"] = round(float(box.r[i]), 4)
            metrics[f"F1_{cls_name}"] = round(float(box.f1[i]), 4)
        except Exception: pass
    return metrics

test_metrics = extract_metrics(test_results, "test")
fitness = round(float(test_results.fitness), 4)
speed = test_results.speed
print("=" * 55)
print(f"  EXPERIMENT : {EXPERIMENT_ID.upper()}")
print("=" * 55)
print(f"  mAP@50      : {test_metrics['mAP50']}")
print(f"  mAP@50-95   : {test_metrics['mAP50_95']}")
print(f"  Precision   : {test_metrics['precision']}")
print(f"  Recall      : {test_metrics['recall']}")
print(f"  F1          : {test_metrics['f1']}")
print(f"  Fitness     : {fitness}")
print(f"  Inference   : {speed['inference']:.2f} ms/img")
print(f"  Preprocess  : {speed['preprocess']:.2f} ms/img")
print(f"  Postprocess : {speed['postprocess']:.2f} ms/img")
for cls in CLASS_NAMES:
    print(f"  AP50 [{cls:<12}]: {test_metrics.get(f'AP50_{cls}', 'N/A')}")
print("=" * 55)


  EXPERIMENT : SCALE_ADAPTIVE_CBAM
  mAP@50      : 0.9785
  mAP@50-95   : 0.6744
  Precision   : 0.956
  Recall      : 0.9395
  F1          : 0.9477
  Fitness     : 0.6744
  Inference   : 5.22 ms/img
  Preprocess  : 1.43 ms/img
  Postprocess : 1.52 ms/img
  AP50 [helmet      ]: 0.9847
  AP50 [non-helmet  ]: 0.9723


## 3 - Save Metrics

In [7]:
test_metrics["fitness"] = fitness
test_metrics["inference_ms_img"] = round(speed["inference"], 2)
test_metrics["preprocess_ms_img"] = round(speed["preprocess"], 2)
test_metrics["postprocess_ms_img"] = round(speed["postprocess"], 2)

csv_path = f"{OUTPUT_DIR}/metrics_{EXPERIMENT_ID}.csv"
df = pd.DataFrame([test_metrics])
df.to_csv(csv_path, index=False)
print(f"[OK] CSV saved -> {csv_path}")

json_path = f"{OUTPUT_DIR}/metrics_{EXPERIMENT_ID}.json"
with open(json_path, "w") as f:
    json.dump([test_metrics], f, indent=2)
print(f"[OK] JSON saved -> {json_path}")

std_csv = f"{OUTPUT_DIR}/test_results.csv"
std_row = {"Model Name": EXPERIMENT_ID, "Precision": test_metrics["precision"],
           "Recall": test_metrics["recall"], "F1": test_metrics["f1"],
           "mAP50": test_metrics["mAP50"], "mAP50_95": test_metrics["mAP50_95"],
           "Fitness": fitness, "Inference_ms_img": round(speed["inference"], 2)}
pd.DataFrame([std_row]).to_csv(std_csv, index=False)
print(f"[OK] Standardized CSV -> {std_csv}")
df

[OK] CSV saved -> C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam/metrics_scale_adaptive_cbam.csv
[OK] JSON saved -> C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam/metrics_scale_adaptive_cbam.json
[OK] Standardized CSV -> C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam/test_results.csv


,experiment,split,mAP50,mAP50_95,precision,recall,f1,AP50_helmet,AP50_95_helmet,P_helmet,...,F1_helmet,AP50_non-helmet,AP50_95_non-helmet,P_non-helmet,R_non-helmet,F1_non-helmet,fitness,inference_ms_img,preprocess_ms_img,postprocess_ms_img
0,scale_adaptive_cbam,test,0.9785,0.6744,0.956,0.9395,0.9477,0.9847,0.6797,0.9676,...,0.9534,0.9723,0.6691,0.9445,0.9395,0.942,0.6744,5.22,1.43,1.52


## 4 - Metrics Bar Chart

In [8]:
metric_keys = ["mAP50","mAP50_95","precision","recall","f1"]
metric_labels = ["mAP@50","mAP@50-95","Precision","Recall","F1"]
test_vals = [test_metrics[k] for k in metric_keys]
x = np.arange(len(metric_labels))
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(x, test_vals, 0.5, color="#DD8452", alpha=0.88)
ax.set_ylim(0, 1.05); ax.set_xticks(x); ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title(f"Test Metrics - {EXPERIMENT_ID.upper()}", fontsize=14, fontweight="bold")
ax.grid(axis="y", linestyle="--", alpha=0.5)
for bar in bars:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x()+bar.get_width()/2, h), xytext=(0,3), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout()
chart_path = f"{OUTPUT_DIR}/metrics_chart_{EXPERIMENT_ID}.png"
plt.savefig(chart_path, dpi=150); plt.show()
print(f"[OK] Chart saved -> {chart_path}")

<Figure size 1000x500 with 1 Axes>

[OK] Chart saved -> C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam/metrics_chart_scale_adaptive_cbam.png


## 5 - Per-Class AP

In [9]:
ap50_vals = [test_metrics.get(f"AP50_{c}", 0) for c in CLASS_NAMES]
ap95_vals = [test_metrics.get(f"AP50_95_{c}", 0) for c in CLASS_NAMES]
x = np.arange(len(CLASS_NAMES)); width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x-width/2, ap50_vals, width, label="AP@50", color="#55A868", alpha=0.88)
b2 = ax.bar(x+width/2, ap95_vals, width, label="AP@50-95", color="#C44E52", alpha=0.88)
ax.set_ylim(0,1.05); ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_ylabel("AP Score", fontsize=12)
ax.set_title(f"Per-Class AP (Test) - {EXPERIMENT_ID.upper()}", fontsize=14, fontweight="bold")
ax.legend(fontsize=11); ax.grid(axis="y", linestyle="--", alpha=0.5)
for bar in [*b1,*b2]:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x()+bar.get_width()/2,h), xytext=(0,3), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout()
pc_path = f"{OUTPUT_DIR}/per_class_ap_{EXPERIMENT_ID}.png"
plt.savefig(pc_path, dpi=150); plt.show()
print(f"[OK] Per-class chart saved -> {pc_path}")

<Figure size 800x500 with 1 Axes>

[OK] Per-class chart saved -> C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam/per_class_ap_scale_adaptive_cbam.png


## 6 - Model Info and Final Summary

In [10]:
info = model.info(verbose=True)
print(f"\n  Weights: {WEIGHTS}")
print("\n" + "=" * 60)
print(f"  FINAL SUMMARY - {EXPERIMENT_ID.upper()}")
print("=" * 60)
for key, label in zip(metric_keys, metric_labels):
    print(f"  {label:<20} {test_metrics[key]:>10.4f}")
print(f"  {'Fitness':<20} {fitness:>10.4f}")
print(f"  {'Inference ms/img':<20} {speed['inference']:>10.2f}")
print("=" * 60)
print(f"\n  Outputs saved to: {OUTPUT_DIR}/")


scale_adaptive_cbam summary (fused): 97 layers, 3,031,932 parameters, 0 gradients, 8.1 GFLOPs



  Weights: C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/models/best (scale_adaptive).pt

  FINAL SUMMARY - SCALE_ADAPTIVE_CBAM
  mAP@50                   0.9785
  mAP@50-95                0.6744
  Precision                0.9560
  Recall                   0.9395
  F1                       0.9477
  Fitness                  0.6744
  Inference ms/img           5.22

  Outputs saved to: C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_big\eval_scale_adaptive_cbam/
